# Titanic Survival Prediction
## Task 1: Cross Validation | Task 2: Best Model + Save/Load with joblib

## 0. Upload the Dataset (Google Colab)
شغّلي الخلية دي وارفعي ملف `train.csv` لما تظهرلك زرارة الرفع.

In [ ]:
from google.colab import files
uploaded = files.upload()  # اختاري train.csv من جهازك


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib
import warnings
warnings.filterwarnings('ignore')


## 2. Load Data

In [ ]:
df = pd.read_csv('train.csv')
print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


## 3. Feature Selection & Preprocessing Setup

هنشيل الأعمدة اللي مالهاش قيمة تنبؤية زي `PassengerId`, `Name`, `Ticket`, `Cabin` (فيها missing كتير جداً).
هنعمل Pipeline كامل للـ preprocessing (Imputation + Encoding + Scaling) عشان يتطبق صح جوه الـ Cross Validation ويمنع Data Leakage.

In [ ]:
# Drop irrelevant / high-missing columns
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_model = df.drop(columns=drop_cols)

X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Pclass', 'Sex', 'Embarked']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


## Task 1: Apply Cross Validation on the Dataset

In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring='accuracy')

print('Cross Validation Accuracy per fold:', cv_scores)
print(f'Mean CV Accuracy: {cv_scores.mean():.4f}')
print(f'Std CV Accuracy:  {cv_scores.std():.4f}')


## Task 2: Try Different Models to Get the Best Accuracy

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results[name] = scores.mean()
    print(f'{name:20s} -> Mean CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')

print()
best_model_name = max(results, key=results.get)
print(f'Best Model (before tuning): {best_model_name} with accuracy {results[best_model_name]:.4f}')


### Hyperparameter Tuning (GridSearchCV) on the Best Candidates

هنعمل tuning لـ Random Forest و SVM و Gradient Boosting عشان نطلع بأحسن accuracy ممكنة.

In [ ]:
param_grids = {
    'Random Forest': {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200, 300],
        'classifier__max_depth': [None, 5, 10, 15],
        'classifier__min_samples_split': [2, 5, 10]
    },
    'SVM': {
        'classifier': [SVC(random_state=42, probability=True)],
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['rbf', 'linear'],
        'classifier__gamma': ['scale', 'auto']
    },
    'Gradient Boosting': {
        'classifier': [GradientBoostingClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__max_depth': [2, 3, 4]
    }
}

base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

best_estimators = {}

for name, grid in param_grids.items():
    print(f'--- Tuning {name} ---')
    search = GridSearchCV(base_pipeline, grid, cv=cv, scoring='accuracy', n_jobs=-1)
    search.fit(X_train, y_train)
    best_estimators[name] = search
    print(f'Best CV Accuracy: {search.best_score_:.4f}')
    print(f'Best Params: {search.best_params_}')
    print()


In [ ]:
best_name = max(best_estimators, key=lambda n: best_estimators[n].best_score_)
best_search = best_estimators[best_name]
best_pipeline = best_search.best_estimator_

print(f'Overall Best Model: {best_name}')
print(f'Best CV Accuracy: {best_search.best_score_:.4f}')
print(f'Best Params: {best_search.best_params_}')


### Evaluate Best Model on the Held-out Test Set

In [ ]:
y_pred = best_pipeline.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)

print(f'Test Set Accuracy: {test_acc:.4f}')
print()
print(classification_report(y_test, y_pred))
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))


### Refit Best Model on Full Dataset (Train + Test) Before Saving

عشان أحسن استفادة من الداتا كلها قبل ما نحفظ الموديل النهائي.

In [ ]:
best_pipeline.fit(X, y)
print('Final pipeline refit on full dataset.')


## Save the Model with joblib

In [ ]:
joblib.dump(best_pipeline, 'titanic_best_model.joblib')
print('Model saved as titanic_best_model.joblib')


## Load the Model with joblib and Test it on Raw Data

In [ ]:
loaded_pipeline = joblib.load('titanic_best_model.joblib')

# Raw data sample (as it would appear before any preprocessing)
raw_sample = pd.DataFrame([
    {'Pclass': 3, 'Sex': 'male', 'Age': 22, 'SibSp': 1, 'Parch': 0, 'Fare': 7.25, 'Embarked': 'S'},
    {'Pclass': 1, 'Sex': 'female', 'Age': 38, 'SibSp': 1, 'Parch': 0, 'Fare': 71.28, 'Embarked': 'C'},
    {'Pclass': 3, 'Sex': 'female', 'Age': 26, 'SibSp': 0, 'Parch': 0, 'Fare': 7.925, 'Embarked': 'S'},
])

predictions = loaded_pipeline.predict(raw_sample)
probabilities = loaded_pipeline.predict_proba(raw_sample)[:, 1]

raw_sample['Survived_Prediction'] = predictions
raw_sample['Survival_Probability'] = probabilities.round(3)
raw_sample


**النتيجة:** الموديل بيقبل الداتا الخام زي ما هي (من غير أي preprocessing يدوي) لأن الـ Pipeline نفسه فيه كل خطوات الـ Imputation, Encoding, و Scaling جواه، فبيتطبقوا أوتوماتيك وقت الـ `.predict()`.

## Download the Saved Model (Google Colab)
شغّلي الخلية دي عشان تنزّلي ملف `titanic_best_model.joblib` على جهازك.

In [ ]:
from google.colab import files
files.download('titanic_best_model.joblib')
